In [ ]:
#Passo 1: parâmetros
import numpy as np

S0 = 100      # preço inicial da ação
K = 100       # strike da opção
T = 1.0       # prazo em anos
r = 0.05      # taxa livre de risco
sigma = 0.20  # volatilidade ao ano
N = 3         # passos da árvore

#Passo 2: fatores da árvore
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1 / u
p = (np.exp(r * dt) - d) / (u - d)
desconto = np.exp(-r * dt)
print(f"u = {u:.4f}  d = {d:.4f}  p = {p:.4f}")

#Passo 3: a árvore de preços
precos = np.zeros((N + 1, N + 1))
for i in range(N + 1):
    for j in range(i + 1):
        precos[j, i] = S0 * u ** (i - j) * d ** j

#Passo 4: valor da opção, de trás para frente
valor = np.zeros((N + 1, N + 1))
valor[:, N] = np.maximum(precos[:, N] - K, 0)

for i in range(N - 1, -1, -1):
    for j in range(i + 1):
        cima = valor[j, i + 1]
        baixo = valor[j + 1, i + 1]
        esperado = p * cima + (1 - p) * baixo
        valor[j, i] = desconto * esperado

print(f"Call europeia (N = {N}): {valor[0, 0]:.4f}")

#Passo 5: tudo em uma função
def call_binomial(S0, K, T, r, sigma, N):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    desconto = np.exp(-r * dt)
    j = np.arange(N + 1)
    v = np.maximum(S0 * u ** (N - j) * d ** j - K, 0)
    for _ in range(N):
        v = desconto * (p * v[:-1] + (1 - p) * v[1:])
    return v[0]

#Passo 6: Black-Scholes e convergência
from math import erf, exp, log, sqrt

def cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

num = log(S0 / K) + (r + sigma**2 / 2) * T
d1 = num / (sigma * sqrt(T))
d2 = d1 - sigma * sqrt(T)
bs = S0 * cdf(d1) - K * exp(-r * T) * cdf(d2)
print(f"Black-Scholes: {bs:.4f}")
for n in (1, 3, 10, 50, 200, 1000):
    c = call_binomial(S0, K, T, r, sigma, n)
    print(f"N = {n:>4}: {c:.4f}")

#Código do gráfico 1
import matplotlib.pyplot as plt
plt.style.use("dark_background")
amarelo = "#FFDF00"
caixa = dict(fc="black", ec=amarelo, boxstyle="round")
fmt = dict(ha="center", va="center", bbox=caixa)

plt.figure(figsize=(7, 4))
for i in range(N + 1):
    for j in range(i + 1):
        y = i - 2 * j
        if i < N:
            plt.plot([i, i + 1], [y, y + 1], "gray")
            plt.plot([i, i + 1], [y, y - 1], "gray")
        txt = f"{precos[j, i]:.1f}\n{valor[j, i]:.2f}"
        plt.text(i, y, txt, **fmt)
plt.axis("off")
plt.show()

#Código do gráfico 2
Ns = range(1, 101)
f = lambda n: call_binomial(S0, K, T, r, sigma, n)
valores = [f(n) for n in Ns]

plt.figure(figsize=(7, 4))
plt.plot(Ns, valores, color=amarelo)
plt.axhline(bs, color="white", linestyle="--")
plt.xlabel("Número de passos (N)")
plt.ylabel("Preço da call")
plt.show()

#Passo 7: a put americana
def put_binomial(S0, K, T, r, sigma, N, americana=False):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    desconto = np.exp(-r * dt)
    j = np.arange(N + 1)
    v = np.maximum(K - S0 * u ** (N - j) * d ** j, 0)
    for i in range(N - 1, -1, -1):
        v = desconto * (p * v[:-1] + (1 - p) * v[1:])
        if americana:
            k = np.arange(i + 1)
            S = S0 * u ** (i - k) * d ** k
            v = np.maximum(v, K - S)
    return v[0]

#Passo 8: comparar e conferir
eur = put_binomial(S0, K, T, r, sigma, 500)
ame = put_binomial(S0, K, T, r, sigma, 500, True)
put_bs = bs - S0 + K * exp(-r * T)

print(f"Put europeia (binomial): {eur:.4f}")
print(f"Put europeia (Black-Scholes): {put_bs:.4f}")
print(f"Put americana (binomial): {ame:.4f}")
print(f"Prêmio de antecipação: {ame - eur:.4f}")

#Código do gráfico 3
S = np.linspace(60, 140, 41)
n = 200
payoff = np.maximum(K - S, 0)
eur = [put_binomial(s, K, T, r, sigma, n) for s in S]
ame = [put_binomial(s, K, T, r, sigma, n, True) for s in S]

plt.figure(figsize=(7, 4))
plt.plot(S, payoff, "--", color="gray")
plt.plot(S, eur, color="white")
plt.plot(S, ame, color=amarelo, linewidth=3)
plt.xlabel("Preço da ação")
plt.ylabel("Valor da put")
plt.show()